# 07. MediaPipe로 가위바위보 판정기 만들기 (실습)

이 실습은 `src/mediapipe/hand_lite/`에 이미 구현된 MediaPipe 기반 가위바위보
인식기를 **직접 재구현**해봅니다. 05강(OpenCV, 2/6)·06강(YOLO, 4/6)과 같은 6장
사진에 돌려서, MediaPipe가 왜 6/6을 만들어내는지 랜드마크 좌표로 직접 확인합니다.

## 학습 목표
- `HandLandmarker`(Tasks API)로 21개 손 랜드마크를 검출할 수 있다.
- TIP·PIP 관절의 y좌표 비교로 검지~소지의 펴짐 여부를 판정할 수 있다.
- 왜 엄지만 "손 크기로 정규화한 거리비"가 필요한지 설명할 수 있다(x좌표 비교의 실패 이유).
- 랜드마크에 "이름"이 있다는 것이 05강의 전통 기법과 근본적으로 다른 점임을 설명할 수 있다.

> 빈칸은 `_____`로 표시돼 있고, 바로 위/옆의 `# [TODO]` 주석이 힌트입니다.
> 원본 구현은 `src/mediapipe/hand_lite/hand_lite/gesture.py`에 있습니다.

In [1]:
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision


def find_workshop_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "yolo" / "rps").is_dir():
            return candidate
    raise FileNotFoundError("워크샵 루트를 찾지 못했습니다")


WORKSHOP = find_workshop_root(Path.cwd())
FIXTURE_DIR = WORKSHOP / "src" / "yolo" / "rps" / "tests" / "fixtures"
MODEL_PATH = WORKSHOP / "src" / "mediapipe" / "hand_lite" / "models" / "hand_landmarker.task"

FIXTURES = ["rock_0", "rock_1", "paper_0", "paper_1", "scissors_0", "scissors_1"]
TRUTH = {
    "rock_0": "ROCK", "rock_1": "ROCK",
    "paper_0": "PAPER", "paper_1": "PAPER",
    "scissors_0": "SCISSORS", "scissors_1": "SCISSORS",
}

# BGR(OpenCV) → RGB(MediaPipe) 변환 후 mp.Image로 감싼다.
mp_images = {}
for name in FIXTURES:
    frame_bgr = cv2.imread(str(FIXTURE_DIR / f"{name}.png"))
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_images[name] = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

print("이미지", len(mp_images), "장 로드 완료")

이미지 6 장 로드 완료


## 0. HandLandmarker 초기화 — IMAGE 모드

정지 이미지 6장을 각각 독립적으로 처리하므로 `RunningMode.IMAGE`를 씁니다(03강에서
본 것처럼 VIDEO 모드는 타임스탬프가 단조 증가해야 하는 제약이 있어 여기서는 불필요합니다).
`hand_lite`가 실제로 쓰는 VIDEO 모드는 웹캠처럼 연속 프레임에서 이전 프레임 추적을
재사용하기 위한 것입니다.

In [2]:
base_options = mp_python.BaseOptions(model_asset_path=str(MODEL_PATH))  # [TODO] 모델 파일 경로 지정
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)  # [TODO] IMAGE 모드가 기본값이므로 running_mode 생략 가능
landmarker = vision.HandLandmarker.create_from_options(options)  # [TODO] 옵션으로 landmarker 생성

result = landmarker.detect(mp_images["rock_0"])
print("검출된 손 개수:", len(result.hand_landmarks))
print("랜드마크 개수:", len(result.hand_landmarks[0]))

검출된 손 개수: 1
랜드마크 개수: 21


I0000 00:00:1785480550.474286  275962 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M3 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1785480550.512029  275964 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785480550.535255  275964 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785480550.595474  275963 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


## 1. 검지~소지 판정 — TIP vs PIP

MediaPipe 좌표계는 정규화 좌표(0~1)이고 **화면 위쪽일수록 y가 작습니다**. 손가락을
펴면 손끝(TIP)이 두 번째 관절(PIP)보다 위(작은 y)에 있습니다. 검지=(5,6,7,8),
중지=(9,10,11,12), 약지=(13,14,15,16), 소지=(17,18,19,20) — 각 튜플은
(MCP, PIP, DIP, TIP) 인덱스입니다.

In [3]:
_INDEX = (5, 6, 7, 8)
_MIDDLE = (9, 10, 11, 12)
_RING = (13, 14, 15, 16)
_PINKY = (17, 18, 19, 20)


def is_extended(landmarks, finger: tuple[int, int, int, int]) -> bool:
    """finger = (MCP, PIP, DIP, TIP) 인덱스. TIP이 PIP보다 위에 있으면 펴진 것."""
    _mcp, pip, _dip, tip = finger
    return landmarks[tip].y < landmarks[pip].y  # [TODO] 손끝(tip)이 두 번째 관절(pip)보다 화면 위쪽(y가 작음)에 있으면 펴진 것


# 검증: rock_0(주먹)에서는 네 손가락 모두 안 펴져 있어야 한다
lm = landmarker.detect(mp_images["rock_0"]).hand_landmarks[0]
assert is_extended(lm, _INDEX) is False
print("is_extended 검증 통과 — rock_0에서 검지가 굽어있음(False)")

is_extended 검증 통과 — rock_0에서 검지가 굽어있음(False)


## 2. 엄지 판정 — 왜 부등호 비교로 안 되는가

최초 구현은 엄지를 "x좌표 비교 + handedness로 부등호 반전"으로 판정했지만, 실사진
6장 검증 결과 **엄지만 0/6**이었습니다. 원인은 사진이 전부 손등이 보이는 각도였고,
주먹을 쥐면 엄지가 접힌 손가락 위에 얹히면서 x좌표만으로는 "펴짐"으로 잘못 읽혔기
때문입니다. 지금은 "엄지 끝이 검지 밑동(INDEX_MCP)에서 얼마나 멀리 떨어졌는가"를
손 크기(손목-중지밑동 거리)로 정규화한 **거리비**로 판정합니다 — 방향(x/y축)에
의존하지 않으므로 손이 뒤집혀도(거울 모드) 안전합니다.

임계값 0.57은 실측(굽음 최대 0.266, 펴짐 최소 0.865)의 중앙값입니다.

In [4]:
_WRIST = 0
_THUMB_TIP = 4
_INDEX_MCP = 5
_MIDDLE_MCP = 9
_THUMB_SPREAD_RATIO_THRESHOLD = 0.57

def _dist(landmarks, a: int, b: int) -> float:
    return math.hypot(landmarks[a].x - landmarks[b].x, landmarks[a].y - landmarks[b].y)  # [TODO] 두 랜드마크 사이 유클리드 거리


def thumb_extended(landmarks, *, threshold: float = _THUMB_SPREAD_RATIO_THRESHOLD) -> bool:
    hand_size = _dist(landmarks, _WRIST, _MIDDLE_MCP)  # [TODO] 손 크기 기준 — 손목~중지밑동 거리
    if hand_size == 0:
        return False
    spread_ratio = _dist(landmarks, _THUMB_TIP, _INDEX_MCP) / hand_size  # [TODO] 엄지끝~검지밑동 거리를 손 크기로 정규화
    return spread_ratio > threshold  # [TODO] 임계값을 넘으면 벌어짐(펴짐)


# 검증: rock_0(주먹)에서는 엄지도 안 펴져 있어야 한다
assert thumb_extended(lm) is False
print("thumb_extended 검증 통과 — rock_0에서 엄지가 접혀있음(False)")

thumb_extended 검증 통과 — rock_0에서 엄지가 접혀있음(False)


## 3. 가위바위보 분류 규칙

주먹(다섯 손가락 전부 굽음)=바위, 다섯 손가락 전부 펴짐=보, 검지+중지가 펴지고
약지+소지가 굽으면=가위(엄지는 무관 — 실제 사람은 가위를 낼 때 엄지를 벌린 채로
냅니다). 그 외 조합은 UNKNOWN입니다.

In [5]:
def classify(landmarks) -> str:
    """가위바위보 판정. 애매하면 UNKNOWN."""
    thumb = thumb_extended(landmarks)
    index = is_extended(landmarks, _INDEX)
    middle = is_extended(landmarks, _MIDDLE)
    ring = is_extended(landmarks, _RING)
    pinky = is_extended(landmarks, _PINKY)

    if not any((thumb, index, middle, ring, pinky)):
        return "ROCK"  # [TODO] 다섯 손가락 전부 굽음
    if all((thumb, index, middle, ring, pinky)):
        return "PAPER"  # [TODO] 다섯 손가락 전부 펴짐
    if index and middle and not ring and not pinky:
        return "SCISSORS"  # [TODO] 검지+중지만 펴짐(엄지 무관)
    return "UNKNOWN"


print("rock_0 분류 결과:", classify(lm))

rock_0 분류 결과: ROCK


## 4. 종합 — 6장 실사진으로 정확도 측정

05강(OpenCV, 2/6)·06강(YOLO, 4/6)과 같은 6장에 돌립니다. **6/6이 나오면 정상입니다.**

In [6]:
correct = 0
for name in FIXTURES:
    result = landmarker.detect(mp_images[name])
    if not result.hand_landmarks:
        predicted = "UNKNOWN"
    else:
        predicted = classify(result.hand_landmarks[0])
    is_correct = predicted == TRUTH[name]
    correct += is_correct
    print(f"{name:12s} 정답={TRUTH[name]:9s} 예측={predicted:9s} {'O' if is_correct else 'X'}")

print(f"\n정확도: {correct}/{len(FIXTURES)}")

rock_0       정답=ROCK      예측=ROCK      O


rock_1       정답=ROCK      예측=ROCK      O


paper_0      정답=PAPER     예측=PAPER     O


paper_1      정답=PAPER     예측=PAPER     O


scissors_0   정답=SCISSORS  예측=SCISSORS  O


scissors_1   정답=SCISSORS  예측=SCISSORS  O

정확도: 6/6


## 5. 왜 MediaPipe는 가위를 맞히는데 OpenCV는 못 맞히는가

05강에서 본 것처럼 OpenCV의 convexity defects는 "골이 2개"라는 사실만 알 뿐 **어느
골이 엄지 쪽인지**는 모릅니다. 반면 MediaPipe는 21개 랜드마크에 각각 이름이 붙어
있어 "엄지(인덱스 4)는 무시하고 검지·중지만 본다"는 규칙을 직접 쓸 수 있습니다.
이 차이가 표현 방식(사전학습된 랜드마크 vs 실루엣 기하)의 근본적인 차이입니다.

---

## AI Pair 섹션

### 1️⃣ Solo — 직접 해보기
`_THUMB_SPREAD_RATIO_THRESHOLD`를 0.3이나 0.8로 바꿔보고 6장 정확도가 어떻게
바뀌는지 확인해보세요. 이 프로젝트 주석에 적힌 "표본 6장뿐이므로 재보정이
필요할 수 있다"는 경고가 왜 있는지 체감할 수 있습니다.

### 2️⃣ Review — AI에게 리뷰 받기
아래 프롬프트를 ChatGPT/Claude에 그대로 복사해서 물어보세요:

> "MediaPipe HandLandmarker의 21개 손 랜드마크 중 엄지 관련 인덱스(1,2,3,4)와
> 검지 관련 인덱스(5,6,7,8)가 왜 그렇게 배치돼 있는지, 그리고 이 랜드마크 순서를
> 이용해서 손가락이 펴졌는지 판정하는 다른 방법(각도 기반 등)도 있는지 알려줘."

### 3️⃣ Debug — 틀린 코드 고치기
아래 코드는 실행되지만 손이 없는 이미지에서 죽습니다. 어디가 문제인지 찾아 고쳐보세요.

In [7]:
# 이 코드는 손이 있는 이미지에서는 잘 동작하지만, 손이 없는 이미지에서는 IndexError로 죽습니다.
def classify_buggy(mp_image):
    result = landmarker.detect(mp_image)
    landmarks = result.hand_landmarks[0]  # 힌트: 손이 검출되지 않으면 이 리스트는 비어 있습니다
    return classify(landmarks)

### 4️⃣ Prompt Card — 더 탐구해보기
- "IMAGE 모드와 VIDEO 모드의 HandLandmarker는 왜 별도 인스턴스로 만들어야 하는지, 하나를 공유하면 어떤 문제가 생기는지"
- "z좌표(손목 기준 상대 깊이)를 이용하면 엄지 판정을 더 안정적으로 만들 수 있을까?"
- "이 6장의 진짜 웹캠 영상(정지 이미지가 아니라)에서도 6/6이 유지될지, 어떤 조건에서 깨질 수 있을지"

---

## 📚 세션 요약

**🎯 핵심 Takeaways**
- MediaPipe 좌표계는 정규화(0~1)돼 있고, 화면 위쪽일수록 y가 작다.
- 랜드마크에 "이름"이 있다는 것(엄지=인덱스 4)이 전통 기법(05강) 대비 MediaPipe의 근본적인 강점이다.
- 부등호(x좌표) 비교는 카메라 각도·거울 모드에 취약하다 — 손 크기로 정규화한 거리비가 더 강건하다.
- 6/6이라는 결과 자체를 "완벽함"으로 착각하면 안 된다 — 표본 6장에서 나온 임계값이라는 한계는 여전히 남는다.